# BRAM-EV variants — one internal mechanism at a time

This notebook **runs no simulation**: it reads the artifacts of a campaign that
has already been produced.

```bash
python main.py run --config experiments/ablation_variants.yaml
```

Where `ablation.ipynb` asks *"what does adding this component buy?"*, this
notebook asks *"does this mechanism have to work the way it does?"*. Each
variant replaces **one single** internal mechanism of the complete method and
compares against `bramev`:

| Variant | Mechanism neutralised | Replaced by |
| --- | --- | --- |
| `bramev_nearest_offer` | multi-criteria utility | the vehicle takes the nearest offer |
| `bramev_fixed_alpha` | alpha heterogeneity | one common alpha (`--alpha-fixed`) |
| `bramev_global_rep` | per-company reputation | a single shared score |
| `bramev_event_score` | duration-weighted score | a flat penalty per event |

The reading direction is **BRAM-EV -> variant**: an unfavourable gap means that
the neutralised mechanism was useful. The plan is declared once, in
`src/experiments/methods.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import pandas as pd
from IPython.display import Image, display

import src.experiments.methods as methods
from src.pipeline import ablation, figures
from src.pipeline.params import CaseParams
from src.pipeline.store import RunStore

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

## Choosing the run

Both `bramev` **and** at least one variant are needed: without the reference,
no gap is computable. `latest_with_methods` takes the most recent run that
contains them, and says what the others contain if it finds none.

In [ ]:
for path in RunStore.list_runs('../results_grid'):
    present = sorted({row['method'] for row in RunStore(path).read_summary()})
    print(f"{path.name}\n    {', '.join(present) or 'no case'}")

In [ ]:
store = RunStore.latest_with_methods(('bramev',) + methods.VARIANTS,
                                     '../results_grid')
params = store.read_params()
manifest = store.read_manifest()

print(store.root)
print(params.describe())
print(f"seed={params.seed} | commit={manifest['git_commit']} | "
      f"cases={manifest['nb_cases_done']}/{manifest['nb_cases_planned']}")
print(f"alpha imposed on the fixed-alpha variants: {params.alpha_fixed}")

## What is actually neutralised

Preliminary check: each variant must differ from `bramev` by **exactly one**
mechanism. These columns come from the flags actually applied during the
simulation (`src/pipeline/tables.py`), not from the declared intent.

In [ ]:
summary = pd.read_csv(store.summary_path)

ORDER = ['bramev'] + list(methods.VARIANTS)
summary['method'] = pd.Categorical(summary['method'], ORDER + [
    m for m in summary['method'].unique() if m not in ORDER], ordered=True)

variants = summary[summary['method'].isin(ORDER)].copy()

plan = (variants[['method', 'method_label', 'method_family', 'offer_choice',
                  'alpha_mode', 'reputation_scope', 'score_weighting',
                  'broadcast', 'reputation', 'adaptation']]
        .drop_duplicates()
        .sort_values('method')
        .set_index('method'))
plan

In [ ]:
fields = ['broadcast', 'reputation', 'adaptation', 'offer_choice',
          'alpha_mode', 'reputation_scope', 'score_weighting']
reference = plan.loc['bramev']

for name in methods.VARIANTS:
    if name not in plan.index:
        print(f"MISSING   {name}")
        continue
    changed = [c for c in fields if plan.loc[name, c] != reference[c]]
    state = 'OK' if len(changed) == 1 else 'ANOMALY'
    print(f"{state:9} {name:22} changed={', '.join(changed):<20} "
          f"mechanism={ablation.VARIANT_MECHANISM.get(name, '?')}")

In [ ]:
# Decomposition computed on the fly from summary.csv. The pipeline persists
# exactly the same tables (`ablation.csv`, `ablation_mean.csv`) and rewrites them
# at every case; recomputing them here makes the notebook usable on a campaign
# still running, interrupted, or predating the ablation study.
detail = pd.DataFrame(ablation.detail_rows(summary.to_dict('records')))
means = pd.DataFrame(ablation.mean_rows(detail.to_dict('records')))

print(f"{len(detail)} gaps computed over "
      f"{detail[['scenario', 'nb_cars']].drop_duplicates().shape[0]} worlds")

## The complete method against each variant

One row per method, averaged over every world of the run. `bramev` is the
reference: read the other rows as gaps to it.

In [ ]:
METRICS = ['exact_satisfaction', 'rate_abs', 'mean_service_rate',
           'slot_waste_rate', 'nb_reservations', 'mean_waiting_time_min',
           'mean_travel_distance_km', 'mean_offers_per_demand',
           'total_ms_mean']

levels = (variants.groupby('method', observed=True)[METRICS]
                  .mean()
                  .rename(index=methods.label)
                  .round(4))
levels

In [ ]:
# Relative gap to bramev, in percent. The sign is raw: the direction proper to
# each metric is applied further down by `improvement`.
reference_values = variants[variants['method'] == 'bramev'][METRICS].mean()
gaps = (variants.groupby('method', observed=True)[METRICS].mean()
        .div(reference_values) - 1.) * 100.
gaps.drop(index='bramev').rename(index=methods.label).round(2)

## Effect of the neutralised mechanism

`ablation_mean.csv`, filtered on `kind == 'variant'`. `share_improved` is the
share of the worlds where **neutralising** the mechanism improves the metric: a
low value is therefore an argument *for* the mechanism.

In [ ]:
print(ablation.render_mean_table(means.to_dict('records')))

In [ ]:
block = means[means['kind'] == 'variant']

effects = block.pivot_table(index=['component', 'to_method'],
                            columns='metric_label',
                            values=['mean_delta_pct', 'share_improved'])
effects.round(3)

In [ ]:
# Verdict per mechanism: over how many (world x metric) does neutralising it
# degrade the result? A high share argues for the mechanism.
block_detail = detail[detail['kind'] == 'variant']

verdict = (block_detail.groupby(['component', 'to_method'])
           .agg(comparisons=('improvement', 'size'),
                neutralising_degrades=('improvement', lambda s: (~s).mean()))
           .round(3)
           .sort_values('neutralising_degrades', ascending=False))
verdict

## Dispersion per variant

As for the ladder, an average may be carried by a single world.

In [ ]:
for metric in ['exact_satisfaction', 'rate_abs', 'mean_service_rate']:
    subset = block_detail[block_detail['metric'] == metric]
    if subset.empty:
        continue
    print(f"\n=== {subset['metric_label'].iloc[0]} — gap to bramev ===")
    stats = (subset.groupby('component')['delta']
                   .agg(['count', 'min', 'median', 'mean', 'max'])
                   .round(6))
    stats['worlds_improved'] = subset.groupby('component')['improvement'].mean().round(3)
    display(stats)

## Checks specific to each variant

A variant may show a zero gap simply because its mechanism was not solicited.
The four checks below separate "useless mechanism" from "mechanism never
tested".

### `bramev_nearest_offer` — did the ranking have anything to decide?

Choosing by utility rather than by distance changes nothing if each demand
receives a single offer. `mean_offers_per_demand` says whether the comparison
has any substance.

In [ ]:
offers = (variants.groupby('method', observed=True)['mean_offers_per_demand']
                  .mean().round(3))
print(offers, end='\n\n')

if offers.get('bramev', 0.) <= 1.:
    print("WARNING: at most one offer per demand on average — the selection "
          "criterion almost never had to choose. A zero gap says nothing about "
          "the multi-criteria utility.")
else:
    print(f"{offers['bramev']:.2f} offer(s) per demand: the ranking did have "
          "something to decide.")

### `bramev_fixed_alpha` — was the heterogeneity really removed?

The `alpha` table carries the trajectory of the profit/risk trade-off per
station. Under `bramev` the alphas are dispersed and converge through
collective learning; under `bramev_fixed_alpha` they must all equal
`alpha_fixed`, which makes the learning inert — that is intended, and it is
what isolates the contribution of the heterogeneity itself.

In [ ]:
scenario, nb_cars = params.scenarios[-1], params.fleet_sizes[-1]

def read_alpha(method):
    path = store.table_path(CaseParams(scenario, nb_cars, method), 'alpha')
    return pd.read_csv(path) if path.is_file() else None

for method in ('bramev', 'bramev_fixed_alpha'):
    table = read_alpha(method)
    if table is None:
        print(f"{method:20} (table missing)")
        continue
    initial = table[table['update_step'] == 0]['alpha']
    final = table[table['update_step'] == table['update_step'].max()]['alpha']
    print(f"{method:20} steps={table['update_step'].max() + 1:2}  "
          f"initial alpha: {initial.min():.3f}-{initial.max():.3f} "
          f"(std {initial.std():.4f})  ->  "
          f"final: {final.min():.3f}-{final.max():.3f} "
          f"(std {final.std():.4f})")

In [ ]:
# Mean trajectory and dispersion, learning step by learning step.
trajectories = {}
for method in ('bramev', 'bramev_fixed_alpha'):
    table = read_alpha(method)
    if table is None:
        continue
    trajectories[method] = table.groupby('update_step')['alpha'].agg(
        mean='mean', std='std', minimum='min', maximum='max')

if trajectories:
    display(pd.concat(trajectories, axis=1).round(4))

### `bramev_global_rep` — did the score become a public good?

Under `bramev`, a vehicle carries one score per company (`score_index` = the
owning company): a bad reputation with one does not carry over to the other.
Under `bramev_global_rep`, every station writes and reads the same slot, so
`score_index` is 0 everywhere.

In [ ]:
for method in ('bramev', 'bramev_global_rep'):
    path = store.table_path(CaseParams(scenario, nb_cars, method), 'stations')
    if not path.is_file():
        print(f"{method:20} (table missing)")
        continue
    stations = pd.read_csv(path)
    print(f"{method:20} score_index={sorted(stations['score_index'].unique())}  "
          f"weighting={sorted(stations['score_weighting'].unique())}  "
          f"companies={sorted(stations['society_id'].unique())}")

In [ ]:
# Expected consequence: a shared score degrades faster for a given vehicle, so
# the stations reject more and serve less.
(variants[variants['method'].isin(['bramev', 'bramev_global_rep'])]
 .groupby('method', observed=True)
 .agg(rejected_demands=('nb_station_level_rejections', 'mean'),
      offers_per_demand=('mean_offers_per_demand', 'mean'),
      reservations=('nb_reservations', 'mean'),
      service_rate=('mean_service_rate', 'mean'))
 .round(3))

### `bramev_event_score` — did the duration really weigh?

A flat penalty differs from a proportional one only if the reserved durations
vary. The mean `d_prop` and its dispersion, read from the acceptance table, say
whether the gap between the two weightings could show at all.

In [ ]:
path = store.table_path(CaseParams(scenario, nb_cars, 'bramev'), 'acceptances')
if path.is_file():
    acceptances = pd.read_csv(path)
    print(f"{len(acceptances)} offers accepted")
    display(acceptances[['distance_km', 'waiting_time_min']].describe().round(3))
else:
    print('acceptances table missing (--no-save-tables?)')

# The ratio of the penalties is exactly d_n between the two weightings (see
# Station.update_car_score): the more dispersed the durations, the more the two
# regimes diverge.
config = next(store.iter_results())['config']
print("\nstrategy points (company):", config['base_points_strategy'])

## Figures

In [ ]:
path = store.figure_path('ablation_variants')
if path.is_file():
    print(path.name)
    display(Image(filename=str(path)))

In [ ]:
figures.fig_ablation_variants(summary.to_dict('records'))

## Health of the run

In [ ]:
print('invariants OK:', bool(summary['invariant_ok'].all()))
print('unresolved reservations:', int(summary['nb_unresolved'].sum()))
print('breakdowns:', int(summary['nb_breakdowns'].sum()))

seen = set()
for result in store.iter_results():
    for message in result['behaviors'].get('diagnostics', []):
        if message not in seen:
            seen.add(message)
            print(f"\n[diagnostic] {message}")